In [6]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [14]:
#file loaded to dataframe
df = pd.read_csv("housing.csv")

#first 5 lines of the table
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [15]:
#preprocessing

#yes/no columns ko 1/0 me change krna
binary_columns = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
'airconditioning', 'prefarea']

def binary_map(x):
    return x.map({'yes':1, 'no':0})

df[binary_columns] = df[binary_columns].apply(binary_map)

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,furnished
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,furnished
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,semi-furnished
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,furnished
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,furnished


In [16]:
# furnishingstatus ko one hot encoding krna
# It'll create new columns like furnishingstatus_furnished, 
# furnishingstatus_semi-furnished, furnishingstatus_unfurnished

status = pd.get_dummies(df['furnishingstatus'], drop_first=True)

# Ab in naye columns ko purane data (df) ke saath jod do
df = pd.concat([df, status], axis=1)


# Ab purane 'furnishingstatus' column ki zaroorat nahi hai 
# (kyunki humne naye bana liye), toh usse hata do
df.drop('furnishingstatus', axis=1, inplace=True)

# tru/false ko 1/0 me convert krna
df[['semi-furnished', 'unfurnished']] = df[['semi-furnished', 'unfurnished']].astype(int)

df.head()


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,semi-furnished,unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,0,0
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,0,0
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1,0
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,0,0
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,0,0


In [17]:
df.info

<bound method DataFrame.info of         price  area  bedrooms  ...  prefarea  semi-furnished  unfurnished
0    13300000  7420         4  ...         1               0            0
1    12250000  8960         4  ...         0               0            0
2    12250000  9960         3  ...         1               1            0
3    12215000  7500         4  ...         1               0            0
4    11410000  7420         4  ...         0               0            0
..        ...   ...       ...  ...       ...             ...          ...
540   1820000  3000         2  ...         0               0            1
541   1767150  2400         3  ...         0               1            0
542   1750000  3620         2  ...         0               0            1
543   1750000  2910         3  ...         0               0            0
544   1750000  3850         3  ...         0               0            1

[545 rows x 14 columns]>

In [18]:
# y = target which we need
y = df.pop('price')

# x = features
X = df


In [19]:
# 70% Training: Model ko sikhane ke liye.
# 30% Testing: Model ka test lene ke liye.

from sklearn.model_selection import train_test_split

# Randomly data ko tod rahe hain (random_state=100 taaki humara result same aaye)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=100)

print("Training Data Size:", X_train.shape)
print("Testing Data Size:", X_test.shape)

Training Data Size: (381, 13)
Testing Data Size: (164, 13)


In [20]:
# Training Model

from sklearn.linear_model import LinearRegression

# new model
lm = LinearRegression()

# training model
lm.fit(X_train, y_train)

print("Model Trained Successfully!")


Model Trained Successfully!


In [22]:
from sklearn.metrics import r2_score

# Prediction on test data
y_pred = lm.predict(X_test)

# Calculate R2 Score
score = r2_score(y_test, y_pred)

print("R2 Score:", score, " Accuracy: ", score*100,"%")

R2 Score: 0.672958274345992  Accuracy:  67.2958274345992 %


In [25]:
import pickle
import os

os.makedirs('model', exist_ok=True)
file_path = os.path.join('model', 'house_model.pkl')
# saving the model
pickle.dump(lm, open(file_path, 'wb'))

print("Saved Model location:", file_path)


Saved Model location: model\house_model.pkl


In [26]:
# --- Step 8: Visualization (Graphs) ---
import matplotlib.pyplot as plt
import seaborn as sns

# Predictions nikal rahe hain (Agar upar nahi nikale to)
y_pred = lm.predict(X_test)

# 1. Actual vs Predicted Graph
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, color='blue') # Blue dots
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r', lw=2) # Red line (Perfect)
plt.title('Actual vs Predicted Prices', fontsize=20)
plt.xlabel('Actual Price (Asli)', fontsize=15)
plt.ylabel('Predicted Price (Model)', fontsize=15)
plt.show()

# 2. Error Graph (Residuals)
plt.figure(figsize=(8,6))
sns.histplot((y_test - y_pred), kde=True, bins=20)
plt.title('Error Distribution', fontsize=20)
plt.xlabel('Error (Difference)', fontsize=15)
plt.show()

ModuleNotFoundError: No module named 'matplotlib'